In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import IntSlider, FloatSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# This interactive notebook explores four performance characteristics of a
# continuous-time low-pass Chebyshev Type I filter:
#
#       1. Group delay τg(ω)
#       2. Loss characteristic A(ω)
#       3. Selectivity Fs
#       4. Spectral performance factor Sαβ
#
# Five parameters can be varied:
#
#       N   : filter order
#       ωp  : passband-edge angular frequency
#       Ap  : maximum passband attenuation / ripple in dB
#       α   : lower attenuation level used for Sαβ
#       β   : higher attenuation level used for Sαβ
#
# The filter is constructed directly with
#
#       scipy.signal.cheby1(..., analog=True)
#
# Ripple parameter:
#
#       ε = sqrt(10^(Ap/10) - 1)
#
# Group delay:
#
#       τg(ω) = -dφ(ω)/dω
#
# Loss characteristic:
#
#       A(ω) = -20 log10|H(jω)|
#
# or equivalently
#
#       A(ω) = 10 log10[1 + ε² TN²(ω/ωp)]
#
# The loss-characteristic panel is intentionally magnified around the
# passband and transition region so that the characteristic Chebyshev
# equiripple behavior can be clearly observed.
#
# Selectivity:
#
#       Fs = -d|H(jω)|/dω evaluated at the -3.0103 dB frequency ωc
#
# Spectral performance factor:
#
#       Sαβ = BWβ / BWα
#
# The attenuation levels α and β affect only the spectral performance factor.
#
# All curves are updated in real time without recreating the figures.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:8px 10px;
    margin:0px 0px 7px 0px;
    font-size:13px;
    line-height:1.45;
    background-color:#f7fbff;
    width:790px;
    max-width:790px;
    box-sizing:border-box;
">
<b>Purpose:</b>
Explore the group delay, loss characteristic, selectivity and spectral
performance factor of a continuous-time Chebyshev Type I low-pass filter.
<br>
<b>Interpretation:</b>
The sliders control N, ω<sub>p</sub> and the passband ripple A<sub>p</sub>.
Increasing N sharpens the transition but produces a more strongly varying group
delay. A<sub>p</sub> controls the equal-ripple behavior in the passband.
The loss panel is magnified around the passband so that these ripples can be
clearly observed. The attenuation levels α and β are used to evaluate the
spectral performance factor S<sub>α</sub><sup>β</sup>.
</div>
""", layout=Layout(width='800px', max_width='800px'))

# ==============================================================================
# CONTROLS
# ==============================================================================

slider_layout = Layout(width='250px')
style_opts = {'description_width':'75px'}

order_slider = IntSlider(min=1, max=10, step=1, value=4, description='Order N:', continuous_update=True, style=style_opts, layout=slider_layout)
wp_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=1.0, description='ωp:', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)
Ap_slider = FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='Ap (dB):', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)
alpha_slider = FloatSlider(min=3.0, max=15.0, step=0.5, value=7.0, description='α (dB):', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)
beta_slider = FloatSlider(min=20.0, max=80.0, step=1.0, value=40.0, description='β (dB):', continuous_update=True, readout=True, readout_format='.0f', style=style_opts, layout=slider_layout)

parameter_title = HTML("""
<div style="
    font-size:14px;
    font-weight:bold;
    margin-top:3px;
    margin-bottom:5px;
">
Filter Parameters:
</div>
""")

info_html = HTML(layout=Layout(width='280px', max_width='280px'))

# ==============================================================================
# FIGURE 1: GROUP DELAY
# ==============================================================================

fig_gd, ax_gd = plt.subplots(figsize=(4.6, 3.0))

gd_line, = ax_gd.plot([], [], 'r-', linewidth=2.0, label='τg(ω)')
wp_gd_line = ax_gd.axvline(1.0, color='black', linestyle=':', linewidth=1.1, label='ωp')
zero_gd_line = ax_gd.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_gd.set_xlabel('Angular Frequency ω (rad/s)', fontsize=9)
ax_gd.set_ylabel('Group Delay τg(ω) (s)', fontsize=9)
ax_gd.set_title('Group Delay', fontsize=11, fontweight='bold', pad=5)
ax_gd.tick_params(axis='both', labelsize=8)
ax_gd.grid(True, linestyle=':', alpha=0.5)
ax_gd.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2, fontsize=7)
ax_gd.set_xlim(0.0, 10.0)
ax_gd.set_ylim(0.0, 80.0)

fig_gd.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.85)
fig_gd.canvas.header_visible = False
fig_gd.canvas.toolbar_visible = False
fig_gd.canvas.resizable = False
fig_gd.canvas.layout.width = '460px'
fig_gd.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 2: LOSS CHARACTERISTIC
# ==============================================================================

fig_loss, ax_loss = plt.subplots(figsize=(4.6, 3.0))

loss_line, = ax_loss.plot([], [], 'r-', linewidth=2.0, label='A(ω)')
wp_loss_line = ax_loss.axvline(1.0, color='black', linestyle=':', linewidth=1.1, label='ωp')
Ap_loss_line = ax_loss.axhline(1.0, color='gray', linestyle='--', linewidth=0.9, label='Ap')

ax_loss.set_xlabel('Angular Frequency ω (rad/s)', fontsize=9)
ax_loss.set_ylabel('Loss A(ω) (dB)', fontsize=9)
ax_loss.set_title('Loss Characteristic', fontsize=11, fontweight='bold', pad=5)
ax_loss.tick_params(axis='both', labelsize=8)
ax_loss.grid(True, linestyle=':', alpha=0.5)
ax_loss.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=7)

fig_loss.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.85)
fig_loss.canvas.header_visible = False
fig_loss.canvas.toolbar_visible = False
fig_loss.canvas.resizable = False
fig_loss.canvas.layout.width = '460px'
fig_loss.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 3: SELECTIVITY
# ==============================================================================

fig_sel, ax_sel = plt.subplots(figsize=(4.6, 3.0))

sel_line, = ax_sel.plot([], [], 'r-', linewidth=2.0, label='Fs(N)')
sel_point, = ax_sel.plot([], [], 'ro', markersize=6, label='Current N')

ax_sel.set_xlabel('Filter Order N', fontsize=9)
ax_sel.set_ylabel('Selectivity Fs', fontsize=9)
ax_sel.set_title('Selectivity', fontsize=11, fontweight='bold', pad=5)
ax_sel.tick_params(axis='both', labelsize=8)
ax_sel.grid(True, linestyle=':', alpha=0.5)
ax_sel.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2, fontsize=7)
ax_sel.set_xlim(1.0, 10.0)
ax_sel.set_xticks(np.arange(1, 11, 1))
ax_sel.set_ylim(0.0, 65.0)

fig_sel.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.85)
fig_sel.canvas.header_visible = False
fig_sel.canvas.toolbar_visible = False
fig_sel.canvas.resizable = False
fig_sel.canvas.layout.width = '460px'
fig_sel.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 4: SPECTRAL PERFORMANCE FACTOR
# ==============================================================================

fig_sp, ax_sp = plt.subplots(figsize=(4.6, 3.0))

sp_line, = ax_sp.plot([], [], 'r-', linewidth=2.0, label='Sαβ(N)')
sp_point, = ax_sp.plot([], [], 'ro', markersize=6, label='Current N')
ideal_line = ax_sp.axhline(1.0, color='gray', linestyle='--', linewidth=0.9, label='Ideal = 1')

ax_sp.set_xlabel('Filter Order N', fontsize=9)
ax_sp.set_ylabel('Spectral Factor Sαβ', fontsize=9)
ax_sp.set_title('Spectral Performance Factor', fontsize=11, fontweight='bold', pad=5)
ax_sp.tick_params(axis='both', labelsize=8)
ax_sp.grid(True, linestyle=':', alpha=0.5)
ax_sp.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=7)
ax_sp.set_xlim(1.0, 10.0)
ax_sp.set_xticks(np.arange(1, 11, 1))
ax_sp.set_ylim(1.0, 20.0)

fig_sp.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.85)
fig_sp.canvas.header_visible = False
fig_sp.canvas.toolbar_visible = False
fig_sp.canvas.resizable = False
fig_sp.canvas.layout.width = '460px'
fig_sp.canvas.layout.height = '305px'

# ==============================================================================
# FIXED FREQUENCY AXIS
# ==============================================================================

omega = np.linspace(0.001, 10.0, 4000)
N_values = np.arange(1, 11)

# ==============================================================================
# AUXILIARY FUNCTIONS
# ==============================================================================

def chebyshev1_selectivity(N, wp, Ap):

    epsilon = np.sqrt(10.0**(Ap / 10.0) - 1.0)

    u = np.arccosh(1.0 / epsilon)

    wc = wp * np.cosh(u / N)

    Fs = N * epsilon * np.cosh(u / N) * np.sinh(u) / (2.0 * np.sqrt(2.0) * wc * np.sinh(u / N))

    return Fs, wc

def chebyshev1_bandwidth(N, wp, Ap, attenuation):

    epsilon = np.sqrt(10.0**(Ap / 10.0) - 1.0)

    argument = np.sqrt(10.0**(attenuation / 10.0) - 1.0) / epsilon

    bandwidth = wp * np.cosh(np.arccosh(argument) / N)

    return bandwidth

# ==============================================================================
# UPDATE FUNCTION
# ==============================================================================

def update_chebyshev1_performance(change=None):

    N = order_slider.value
    wp = wp_slider.value
    Ap = Ap_slider.value
    alpha = alpha_slider.value
    beta = beta_slider.value

    # --------------------------------------------------------------------------
    # RIPPLE PARAMETER
    # --------------------------------------------------------------------------

    epsilon = np.sqrt(10.0**(Ap / 10.0) - 1.0)

    # --------------------------------------------------------------------------
    # CHEBYSHEV TYPE I TRANSFER FUNCTION
    # --------------------------------------------------------------------------

    b, a = signal.cheby1(N, Ap, wp, btype='low', analog=True, output='ba')

    # --------------------------------------------------------------------------
    # FREQUENCY RESPONSE
    # --------------------------------------------------------------------------

    _, H = signal.freqs(b, a, worN=omega)

    magnitude = np.abs(H)

    phase = np.unwrap(np.angle(H))

    # --------------------------------------------------------------------------
    # GROUP DELAY
    # --------------------------------------------------------------------------

    group_delay = -np.gradient(phase, omega)

    # --------------------------------------------------------------------------
    # LOSS CHARACTERISTIC
    # --------------------------------------------------------------------------

    magnitude_safe = np.maximum(magnitude, 1e-15)

    loss = -20.0 * np.log10(magnitude_safe)

    # --------------------------------------------------------------------------
    # SELECTIVITY
    # --------------------------------------------------------------------------

    Fs, wc = chebyshev1_selectivity(N, wp, Ap)

    Fs_values = np.array([chebyshev1_selectivity(n, wp, Ap)[0] for n in N_values])

    # --------------------------------------------------------------------------
    # SPECTRAL PERFORMANCE FACTOR
    # --------------------------------------------------------------------------

    BW_alpha = chebyshev1_bandwidth(N, wp, Ap, alpha)

    BW_beta = chebyshev1_bandwidth(N, wp, Ap, beta)

    S = BW_beta / BW_alpha

    S_values = np.array([chebyshev1_bandwidth(n, wp, Ap, beta) / chebyshev1_bandwidth(n, wp, Ap, alpha) for n in N_values])

    # --------------------------------------------------------------------------
    # UPDATE GROUP DELAY
    # --------------------------------------------------------------------------

    gd_line.set_data(omega, group_delay)

    wp_gd_line.set_xdata([wp, wp])

    ax_gd.set_xlim(0.0, 10.0)

    ax_gd.set_ylim(0.0, 80.0)

    # --------------------------------------------------------------------------
    # UPDATE LOSS CHARACTERISTIC
    #
    # Magnified passband/transition view
    # --------------------------------------------------------------------------

    loss_line.set_data(omega, loss)

    wp_loss_line.set_xdata([wp, wp])

    Ap_loss_line.set_ydata([Ap, Ap])

    ax_loss.set_xlim(0.0, 1.2 * wp)

    ax_loss.set_ylim(0.0, max(2.0, 2.0 * Ap))

    # --------------------------------------------------------------------------
    # UPDATE SELECTIVITY
    # --------------------------------------------------------------------------

    sel_line.set_data(N_values, Fs_values)

    sel_point.set_data([N], [Fs])

    ax_sel.set_xlim(1.0, 10.0)

    ax_sel.set_ylim(0.0, 65.0)

    # --------------------------------------------------------------------------
    # UPDATE SPECTRAL PERFORMANCE FACTOR
    # --------------------------------------------------------------------------

    sp_line.set_data(N_values, S_values)

    sp_point.set_data([N], [S])

    ax_sp.set_xlim(1.0, 10.0)

    ax_sp.set_ylim(1.0, 20.0)

    # --------------------------------------------------------------------------
    # INFORMATION PANEL
    # --------------------------------------------------------------------------

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px 9px;
        margin-top:9px;
        font-size:12px;
        line-height:1.60;
        background:white;
        width:275px;
        box-sizing:border-box;
    ">

    <div>
        <b>Filter:</b>
        <span style="color:#0066cc;">Chebyshev Type I low-pass</span>
    </div>

    <div>
        <b>Order N:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Passband edge:</b>
        <span style="color:#0066cc;">ωp = {wp:.2f} rad/s</span>
    </div>

    <div>
        <b>Passband ripple:</b>
        <span style="color:#0066cc;">Ap = {Ap:.2f} dB</span>
    </div>

    <div>
        <b>Ripple parameter:</b>
        <span style="color:#0066cc;">ε = {epsilon:.6f}</span>
    </div>

    <div>
        <b>3-dB cutoff:</b>
        <span style="color:#0066cc;">ωc = {wc:.6f} rad/s</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Performance quantities:</b>
    </div>

    <div>
        <b>Selectivity:</b>
        <span style="color:#0066cc;">Fs = {Fs:.6f}</span>
    </div>

    <div>
        <b>Attenuation levels:</b>
        <span style="color:#0066cc;">α = {alpha:.1f} dB, β = {beta:.1f} dB</span>
    </div>

    <div>
        <b>BWα:</b>
        <span style="color:#0066cc;">{BW_alpha:.6f} rad/s</span>
    </div>

    <div>
        <b>BWβ:</b>
        <span style="color:#0066cc;">{BW_beta:.6f} rad/s</span>
    </div>

    <div>
        <b>Spectral factor:</b>
        <span style="color:#0066cc;">Sαβ = {S:.6f}</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Observation:</b><br>
        Increasing N sharpens the transition and increases selectivity.
        The passband exhibits equal-ripple loss between 0 and Ap.
        The magnified loss panel makes these oscillations directly visible.
        Increasing N also drives Sαβ toward its ideal value 1.
    </div>

    </div>
    """

    # --------------------------------------------------------------------------
    # REDRAW EXISTING FIGURES ONLY
    # --------------------------------------------------------------------------

    fig_gd.canvas.draw_idle()
    fig_loss.canvas.draw_idle()
    fig_sel.canvas.draw_idle()
    fig_sp.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

order_slider.observe(update_chebyshev1_performance, names='value')
wp_slider.observe(update_chebyshev1_performance, names='value')
Ap_slider.observe(update_chebyshev1_performance, names='value')
alpha_slider.observe(update_chebyshev1_performance, names='value')
beta_slider.observe(update_chebyshev1_performance, names='value')

# ==============================================================================
# LAYOUT: 2 x 2 FIGURE GRID
# ==============================================================================

controls = VBox([parameter_title, order_slider, wp_slider, Ap_slider, alpha_slider, beta_slider, info_html], layout=Layout(width='290px', min_width='290px', max_width='290px', flex='0 0 290px', align_items='flex-start'))

top_row = HBox([fig_gd.canvas, fig_loss.canvas], layout=Layout(width='930px', align_items='flex-start', justify_content='flex-start'))

bottom_row = HBox([fig_sel.canvas, fig_sp.canvas], layout=Layout(width='930px', align_items='flex-start', justify_content='flex-start'))

plot_grid = VBox([top_row, bottom_row], layout=Layout(width='930px', align_items='flex-start', justify_content='flex-start'))

main_layout = HBox([controls, plot_grid], layout=Layout(width='1220px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# INITIALIZE DATA
# ==============================================================================

update_chebyshev1_performance()

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)
display(main_layout)